# Data Visualizations

In [2]:
# import packages 
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import font_manager
import matplotlib.dates as mdates
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pywaffle import Waffle
from matplotlib.ticker import FuncFormatter

In [ ]:
# colors 

'#f5f0e8'   # background
'#1a1a1a'   # body
'#06395b'
'#0c629b'
"#458dbd"
"#adddfc"
'#1a5c3a'
"#22794c"
"#67a384"
"#b4d9c6"
"#c48b22"
"#e3b96b"
"#f7dfb3"

color1 = ['#06395b', "#458dbd", "#adddfc", 'rgba(6, 57, 91, 0.3)', 'rgba(69, 141, 189, 0.3)' ]
color2 = ['#1a5c3a', "#4f916f", "#b4d9c6", 'rgba(79, 145, 111, 0.3)', 'rgba(180, 217, 198, 0.3)' ]
color3 = ["#c48b22", "#e3b96b", "#f7dfb3", 'rgba(196, 139, 34, 0.3)', 'rgba(227, 185, 107, 0.3)' ]

In [45]:
# theme 
sns.set_theme(
    style="whitegrid",
    rc={
        "figure.facecolor": "none",
        "axes.facecolor": "none",
        "axes.titlesize": 18,
        "axes.labelsize": 16,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 12,
        "figure.titlesize": 20
    }
)

font_manager.fontManager.addfont("PlayfairDisplay-VariableFont_wght.ttf")

## Overall Percent Spending used on Food

Food personal consumption expenditures / total personal consumption expenditures over time = what percent of spending is used for food over time in general


In [6]:
# import data

food_pce = pd.read_csv("../data/total/DFXARC1M027SBEA.csv", low_memory=False)
pce = pd.read_csv("../data/total/PCE.csv", low_memory=False)

In [23]:
# clean data 

# filter to 1960 and later 
food_pce_clean = food_pce.loc[food_pce['observation_date']>='1960-01-01']
pce_clean = pce.loc[pce['observation_date']>='1960-01-01']

# rename 
food_pce_clean = food_pce_clean.rename(columns={'DFXARC1M027SBEA':'food_pce'})
pce_clean = pce_clean.rename(columns={'PCE':'pce'})

# merge 
percent_food_expend = food_pce_clean.merge(pce_clean, how='outer')

# divide to get percentage
percent_food_expend['percent'] = (percent_food_expend['food_pce']/percent_food_expend['pce']) *100

In [91]:
# line chart of food as a percent of personal consumption expenditures

# data 
y = percent_food_expend["percent"]
x = percent_food_expend["observation_date"]

# figure
fig = go.Figure()

# line plot 
fig.add_trace(
    go.Scatter(
        x=x,
        y=y,
        mode="lines",
        name="",
        line=dict(
            color='#67a384',
            width=4
        ),
        fill='tozeroy',
        fillcolor='rgba(103, 163, 132, 0.18)',
        hovertemplate=
        "Food Percentage: %{y:.1f}%<extra></extra>"
    )
)

# title and layout 
fig.update_layout(
    template="plotly_dark",

    title=dict(
        text="Food Expenditure as a Share of Total Personal Consumption Expenditures",
        x=0.02,
        xanchor="left",
        font=dict(
            family="Playfair Display",
            size=24
        )
    ),

    font=dict(
        family="Source Sans 3",
        color="#f5f0e8",
        size=14
    ),

    paper_bgcolor="#1a1a1a",
    plot_bgcolor="#1a1a1a",

    margin=dict(l=60, r=40, t=80, b=60),

    hovermode="x unified",

    xaxis=dict(
        title="Date",
        showgrid=False,
        tickfont=dict(size=14),
        zeroline=False,
        ticklabelstandoff=10
    ),

    yaxis=dict(
        title="Percentage",
        range=[0, 100],
        ticksuffix="%",
        tickfont=dict(size=14),
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    )
)

# endpoint annotation
fig.add_annotation(
    x=x.iloc[-1],
    y=y.iloc[-1],
    text=f"{y.iloc[-1]:.1f}%",
    showarrow=False,
    xshift=20,
    font=dict(
        color="#f7dfb3",
        size=14
    )
)

fig.show()

In [92]:
# save fig 
fig.write_html(f"../website/charts/percent_food_expenditure.html", include_plotlyjs="cdn", full_html=False)

## Food CPI vs General CPI 

Food cpi vs general cpi  = whether food prices have inflated faster or slower than everything else in the economy over time / is food getting more expensive relative to other things Americans buy?

In [121]:
# import data 

food_cpi = pd.read_csv("../data/total/CPIUFDSL.csv", low_memory=False)
general_cpi = pd.read_csv("../data/total/CPIAUCSL.csv", low_memory=False)

In [122]:
# clean data 

# filter to 1960 and later 
food_cpi_clean = food_cpi.loc[food_cpi['observation_date']>='1960-01-01']
general_cpi_clean = general_cpi.loc[general_cpi['observation_date']>='1960-01-01']

# rename 
food_cpi_clean = food_cpi_clean.rename(columns={'CPIUFDSL':'food_cpi'})
general_cpi_clean = general_cpi_clean.rename(columns={'CPIAUCSL':'total_cpi'})

# merge 
both_cpi = food_cpi_clean.merge(general_cpi_clean, how='outer')

# reindex to January 1960 = 100
base_food = both_cpi.loc[both_cpi['observation_date']=="1960-01-01", "food_cpi"].values
base_all = both_cpi.loc[both_cpi['observation_date']=="1960-01-01", "total_cpi"].values

both_cpi['food_cpi_reindex'] = (both_cpi['food_cpi'] / base_food) * 100
both_cpi['total_cpi_reindex'] = (both_cpi['total_cpi'] / base_all) * 100

# drop na 
both_cpi = both_cpi.dropna()

In [132]:
# line chart of food as a percent of personal consumption expenditures

# data 
x = both_cpi["observation_date"]
food_cpi = both_cpi["food_cpi_reindex"]
total_cpi = both_cpi["total_cpi_reindex"]

# figure
fig = go.Figure()

# line plot 
fig.add_trace(
    go.Scatter(
        x=x,
        y=total_cpi,
        mode="lines",
        name="All Items",
        line=dict(
            color="#458dbd",
            width=4
        ),
        hovertemplate=
        "All Items CPI: %{y:.1f}<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=food_cpi,
        mode="lines",
        name="Food",
        line=dict(
            color='#67a384',
            width=4
        ),
        hovertemplate=
        "Food CPI: %{y:.1f}<extra></extra>"
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=total_cpi,
        mode="lines",
        name="All Items",
        line=dict(
            color="#458dbd",
            width=4
        ),
        fill='tonexty',
        fillcolor='rgba(214, 162, 58, 0.3)',
        hoverinfo='skip',
        showlegend=False
    )
)

# title and layout 
fig.update_layout(
    template="plotly_dark",

    title=dict(
        text="Food vs. Overall Price Inflation",
        x=0.02,
        xanchor="left",
        font=dict(
            family="Playfair Display",
            size=24
        )
    ),

    font=dict(
        family="Source Sans 3",
        color="#f5f0e8",
        size=14
    ),

    paper_bgcolor="#1a1a1a",
    plot_bgcolor="#1a1a1a",

    margin=dict(l=60, r=40, t=80, b=60),

    hovermode="x unified",

    xaxis=dict(
        title="Date",
        showgrid=False,
        tickfont=dict(size=14),
        zeroline=False,
        ticklabelstandoff=5
    ),

    yaxis=dict(
        title="Consumer Price Index for All Urban Customers",
        tickfont=dict(size=14),
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    ),

    legend=dict(
        orientation="h",
        yanchor="top",
        y=0.15,
        xanchor="right",
        x=1,
        itemclick=False,
        itemdoubleclick=False
    )
)

fig.show()

In [133]:
# save fig 
fig.write_html(f"../website/charts/food_vs_overall_inflation.html", include_plotlyjs="cdn", full_html=False)

## Food at Home vs Food Away from Home

Food at home vs food away from home CPIs over time = whether eating at home vs eating out was cheaper at certain times 

In [135]:
# import data 

food_home_cpi = pd.read_csv("../data/categories/CUUR0000SAF11.csv", low_memory=False)
food_away_cpi = pd.read_csv("../data/categories/CUUR0000SEFV.csv", low_memory=False)

In [136]:
# clean data 

# filter to 1960 and later 
food_home_cpi_clean = food_home_cpi.loc[food_home_cpi['observation_date']>='1960-01-01']
food_away_cpi_clean = food_away_cpi.loc[food_away_cpi['observation_date']>='1960-01-01']

# rename 
food_home_cpi_clean = food_home_cpi_clean.rename(columns={'CUUR0000SAF11':'food_home_cpi'})
food_away_cpi_clean = food_away_cpi_clean.rename(columns={'CUUR0000SEFV':'food_away_cpi'})

# merge 
both_cpi = food_home_cpi_clean.merge(food_away_cpi_clean, how='outer')

# reindex to January 1960 = 100
base_home = both_cpi.loc[both_cpi['observation_date']=="1960-01-01", "food_home_cpi"].values
base_away = both_cpi.loc[both_cpi['observation_date']=="1960-01-01", "food_away_cpi"].values

both_cpi['food_home_cpi_reindex'] = (both_cpi['food_home_cpi'] / base_home) * 100
both_cpi['food_away_cpi_reindex'] = (both_cpi['food_away_cpi'] / base_away) * 100

# drop na 
both_cpi = both_cpi.dropna()

In [139]:
# line chart of food at home vs food away from home cpis 

# data 
x = both_cpi["observation_date"]
food_home_cpi = both_cpi["food_home_cpi_reindex"]
food_away_cpi = both_cpi["food_away_cpi_reindex"]

# figure
fig = go.Figure()

# line plot 
fig.add_trace(
    go.Scatter(
        x=x,
        y=food_home_cpi,
        mode="lines",
        name="Food at Home",
        line=dict(
            color='#0c629b',
            width=4
        ),
        hovertemplate=
        "Food at Home CPI: %{y:.1f}<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=food_away_cpi,
        mode="lines",
        name="Food Away from Home",
        line=dict(
            color="#22794c",
            width=4
        ),
        hovertemplate=
        "Food Away from Home CPI: %{y:.1f}<extra></extra>"
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=food_home_cpi,
        mode="lines",
        name="Food at Home",
        line=dict(
            color='#0c629b',
            width=4
        ),
        fill='tonexty',
        fillcolor='rgba(214, 162, 58, 0.3)',
        hoverinfo='skip',
        showlegend=False
    )
)

# title and layout 
fig.update_layout(
    template="plotly_dark",

    title=dict(
        text="Food at Home vs. Away from Home Price Inflation",
        x=0.02,
        xanchor="left",
        font=dict(
            family="Playfair Display",
            size=24
        )
    ),

    font=dict(
        family="Source Sans 3",
        color="#f5f0e8",
        size=14
    ),

    paper_bgcolor="#1a1a1a",
    plot_bgcolor="#1a1a1a",

    margin=dict(l=60, r=40, t=80, b=60),

    hovermode="x unified",

    xaxis=dict(
        title="Date",
        showgrid=False,
        tickfont=dict(size=14),
        zeroline=False,
        ticklabelstandoff=5
    ),

    yaxis=dict(
        title="Consumer Price Index for All Urban Customers",
        tickfont=dict(size=14),
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    ),

    legend=dict(
        orientation="h",
        yanchor="top",
        y=0.15,
        xanchor="right",
        x=1,
        itemclick=False,
        itemdoubleclick=False
    )
)

fig.show()

In [140]:
# save fig 
fig.write_html(f"../website/charts/home_vs_away.html", include_plotlyjs="cdn", full_html=False)